# Sohn et al. (2019), Bayesian computation through cortical latent dynamics, on dandiset 000130

Paper: Sohn, Narain, Meirhaeghe and Jazayeri (2019), "Bayesian computation through cortical latent
dynamics", Neuron 103:934-947, doi:10.1016/j.neuron.2019.06.012. The claims file
(`claims/sohn-2019.yaml`) was written from the accepted author manuscript in PubMed Central
(PMC6805134): Results, figure legends and STAR Methods, without the supplementary tables.

Dataset: DANDI:000130 ("DMFC_RSG", prepared for the Neural Latents Benchmark), version
0.241017.1448, the version this attempt was asked to use; it is also the most recent published
version, and the paper cites no version. Files are streamed from the archive with `remfile`; nothing
is written to disk. The dataset holds one session (2016-12-11) of monkey H (Haydn): a train file with
the trial table and the spikes of 54 units, and a test file with 283 further trials whose task
variables are withheld.

A "dataset" in the paper and here is one effector (Eye, Hand) and target direction (Left, Right): 4 in
this session, 8 in the paper (two animals). Loading and measures are in `rsg.py`, which says where each
parameter comes from. The GPFA claims are in the second notebook, `gpfa_sohn_2019.ipynb`.

| claim | tested here |
|---|---|
| beh-slopes, beh-overlap-prior, beh-sigmoid-increments, beh-long-more-bias, beh-bayes-model | tested |
| glm-prior-ts, dfr-overlap-support, pca-dimensionality | tested |
| encoding-r2, encoding-bayes-beats-linear, short-in-long-control, neural-long-more-bias | tested |
| speed-vs-projection, initial-speed-biased, speed-predicts-tp | tested |
| alt-speed-unmodulated, alt-transient-parallel, alt-threshold-monotonic | tested |
| gpfa-variance-inverted-u, gpfa-mean-sigmoid | tested in `gpfa_sohn_2019.ipynb` |
| rnn-curved-manifold, rnn-perturbation | blocked: the trained networks are not in the dataset nor public |
| monkey-g | blocked: monkey G is not in the dataset |

In [ ]:
import os
os.environ.setdefault("OMP_NUM_THREADS", "1")
import time, resource, warnings
import numpy as np
import pandas as pd
import yaml
import matplotlib.pyplot as plt
import statsmodels.api as sm
from scipy import stats
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
import rsg

warnings.filterwarnings("ignore", category=RuntimeWarning)
T0 = time.time()
DATASET = f"dandi:{rsg.DANDISET}/{rsg.VERSION}"
CLAIMS = {c["id"]: c for c in yaml.safe_load(open("../../claims/sohn-2019.yaml"))["claims"]}
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)


def expected(cid):
    return CLAIMS[cid].get("expected", [])


def within(value, e):
    return abs(value - e["value"]) <= e["tolerance"] + 1e-12


def quote(cid):
    c = CLAIMS[cid]
    print(f"[{cid}] {c['statement']}\nCriterion: {c.get('criterion', '(none)')}")
    for e in expected(cid):
        print(f"  expected {e['quantity']}: {e['value']} +- {e['tolerance']}")

## Data check

The file list and metadata of the pinned version, then every column of the trial and unit tables of
both files.

In [ ]:
assets = rsg.list_assets()
print(DATASET)
print(assets.to_string())
sess = rsg.load_session(rsg.TRAIN_PATH)
test = rsg.load_session(rsg.TEST_PATH)
nwb = sess["nwb"]
print("\nsession:", nwb.session_id, nwb.session_start_time, "|", nwb.session_description)
print("subject:", nwb.subject.subject_id, nwb.subject.species, nwb.subject.age, nwb.subject.sex)
print("related publications:", list(nwb.related_publications))
print("\ntrial table columns and descriptions (train file):")
for c in nwb.trials.colnames:
    print(f"  {c}: {nwb.trials[c].description}")
print("unit table columns:", sess["nwb"].units.colnames)
el = nwb.electrodes.to_dataframe()
print("electrodes:", len(el), "in groups", el.group_name.value_counts().to_dict(),
      "| location", el.location.unique(), "| x, y, z all NaN:", bool(el[["x", "y", "z"]].isna().all().all()))
print("acquisition:", list(nwb.acquisition), "| processing:", list(nwb.processing))
print("\ntest file trial columns:", test["trials"].columns.tolist(), "|", len(test["trials"]), "trials,",
      len(test["units"]), "units (", int(test["units"].heldout.sum()), "held out )")

In [ ]:
tr = rsg.label_trials(sess["trials"])
good = rsg.good_trials(sess["trials"])
units = sess["units"]
ui = rsg.select_units(sess, good)
spikes = [sess["spikes"][i] for i in ui]

print("== trials (train file)")
print("all:", len(tr), "| split:", tr.split.value_counts().to_dict())
print("with Ready:", tr.ready_time.notna().sum(), "| with Set:", tr.set_time.notna().sum(),
      "| with Go:", tr.go_time.notna().sum(), "| tp <= 0:", int((tr.tp <= 0).sum()),
      "| is_outlier:", int(tr.is_outlier.sum()))
print("completed (Set and Go, tp > 0):", int((tr.set_time.notna() & tr.go_time.notna() & (tr.tp > 0)).sum()),
      "| of them flagged is_outlier:", int((tr.set_time.notna() & tr.go_time.notna() & (tr.tp > 0) & tr.is_outlier).sum()),
      "| analysed (completed, not outlier):", len(good))
print("\nanalysed trials by dataset, prior and ts:")
print(pd.crosstab([good.effector, good.direction, good.prior], good.ts))
print("fewest analysed trials in any of the 40 conditions:",
      good.groupby(["prior", "ts", "effector", "direction"]).size().min())

print("\n== ranges and units of the variables used")
for c, unit in [("ts", "ms"), ("tp", "ms")]:
    print(f"{c} ({unit}): {good[c].min():.1f} to {good[c].max():.1f}; ts values {sorted(good.ts.unique())}" if c == "ts"
          else f"{c} ({unit}): {good[c].min():.1f} to {good[c].max():.1f}")
print("theta values:", sorted(tr.theta.unique()), "| is_short x iti (ms):")
print(pd.crosstab(tr.is_short, tr.iti))
dts = (good.set_time - good.ready_time) * 1000 - good.ts
dtp = (good.go_time - good.set_time) * 1000 - good.tp
print("(set_time - ready_time) in ms minus ts: median %.1f, 1st to 99th percentile %.1f to %.1f, range %.1f to %.1f"
      % (dts.median(), *np.percentile(dts, [1, 99]), dts.min(), dts.max()))
print("(go_time - set_time) in ms minus tp:    median %.1f, 1st to 99th percentile %.1f to %.1f, range %.1f to %.1f"
      % (dtp.median(), *np.percentile(dtp, [1, 99]), dtp.min(), dtp.max()))

print("\n== units")
print("train file:", len(units), "units,", int(units.heldout.sum()), "flagged heldout; observed",
      f"{units.observed_s.iloc[0]:.1f} s in", len(sess["obs"][0]), "intervals:", np.round(sess["obs"][0], 1).tolist())
print("all units share the same observation intervals:", all(np.array_equal(o, sess["obs"][0]) for o in sess["obs"]))
print("spike times sorted in every unit:", all(np.all(np.diff(s) >= 0) for s in sess["spikes"]))
print("mean rates (spikes/s): %.2f to %.2f; units >= 1 spike/s: %d" % (units.rate.min(), units.rate.max(), (units.rate >= 1).sum()))
print("units selected (>= 1 spike/s and >= 5 trials in each of the 40 conditions):", len(ui))
print("test file:", len(test["units"]), "units, ids a subset of the train file's:",
      set(test["units"].id) <= set(units.id), "| observed", f"{test['units'].observed_s.iloc[0]:.1f} s")

Checks of the description against the values:

- Trials: 1160 in the train file, of which 1026 are completed (Set and Go present, tp > 0) and 983 are
  completed and not flagged `is_outlier`. Every one of the 40 conditions has at least 14 analysed
  trials, so the unit rule's 5 trials per condition removes no unit; the rate rule keeps 48 of 54 units.
- `ts` takes the 9 values of the two priors, and `set_time - ready_time` matches it to within 23.5 ms
  (median 0), consistent with a 60 Hz display.
- `tp` is longer than `go_time - set_time` in every analysed trial, by a median of 17.9 ms (0.1 to
  200.6 ms). The files do not say which events `tp` is measured between. The behavioural claims use
  `tp`, as the claims file says; the neural alignment uses the event times.
- `iti`: the description says 500 ms for the Short prior and 1220 ms for Long; the values are the
  reverse (Short 1220 in all 577 Short trials, Long 500 in all 583), which is what the paper's Methods
  give. The data are trusted; this is recorded as a finding.
- `theta` is 0 or 180 and `is_eye` is boolean, as described. Electrode positions are all NaN and the
  unit table has no electrode column, so units cannot be placed on the three probes, and the files
  do not say which units are single units and which multi-units.
- The test file holds 283 trials of 1.7 s with only `go_time` (at 1.5 s into each), for 40 units; their
  prior, ts, effector and direction are withheld, so they cannot enter any analysis here.

### Recordings deposited twice

The test file's trials could repeat segments of the train file. For the test unit with the most spikes,
every 50th run of five successive inter-spike intervals is searched for in the same unit's train-file
spike train (to 0.1 ms).

In [ ]:
tid = test["units"].id.values
k = int(np.argmax(test["units"].n_spikes.values))
j = int(np.flatnonzero(units.id.values == tid[k])[0])
isi_train = np.round(np.diff(sess["spikes"][j]), 4)
windows = np.lib.stride_tricks.sliding_window_view(isi_train, 5)
starts = range(0, len(test["spikes"][k]) - 6, 50)
found = sum(bool(np.any(np.all(windows == np.round(np.diff(test["spikes"][k][i:i + 6]), 4), axis=1))) for i in starts)
print(f"unit {tid[k]}: {len(starts)} test-file patterns searched, {found} found in the train file")
gaps = [(a[1], b[0]) for a, b in zip(sess["obs"][0][:-1], sess["obs"][0][1:])]
print("gaps between the train file's observation intervals (s):", [round(b - a, 3) for a, b in gaps])
print("test-file trials: start", test["trials"].start_time.min(), "to", test["trials"].stop_time.max(),
      "s, each", np.unique(np.round(test["trials"].stop_time - test["trials"].start_time, 3)), "s long, Go at",
      np.unique(np.round(test["trials"].go_time - test["trials"].start_time, 3)), "s")

### Do spikes share the clock of the trial table?

Two checks. The target direction is randomised trial by trial (Methods, "Trial structure"), so it can
be decoded from spike counts around Go only when the spikes and the trial times share a clock; with the
trial times shifted by 20 s or more, the labels belong to other trials and decoding falls to chance.
Second, the population rate around Set, at the true times and 2 s later.

In [ ]:
y = (good.direction == "Left").values
for shift in [0.0, 20.0, -20.0, 60.0]:
    C = rsg.bin_counts(spikes, good.go_time.values + shift, -300, 100, bin_ms=400)[:, 0, :]
    acc = cross_val_score(LogisticRegression(max_iter=2000), np.log1p(C), y, cv=5).mean()
    print(f"direction decoded from counts Go-300 to Go+100 ms, times shifted by {shift:+.0f} s: accuracy {acc:.3f} (chance {max(y.mean(), 1 - y.mean()):.3f})")
for name, col in [("Set", "set_time"), ("Ready", "ready_time")]:
    for shift in [0.0, 2.0]:
        R = rsg.bin_counts(spikes, good[col].values + shift, -200, 400, bin_ms=50).mean(axis=(0, 2)) * 20
        print(f"mean rate around {name} (+{shift:.0f} s), 50 ms bins from -200 ms (spikes/s):", np.round(R, 2))

The target direction is decoded from counts around Go with accuracy 0.761 at the true times and 0.477
to 0.517 with the times shifted by 20 or 60 s (chance 0.510). The mean population rate rises 100 to
200 ms after Set (from 9.7 to 13.6 spikes/s) and after Ready (from 6.7 to 10.9) at the true times, and
stays flat with the times shifted by 2 s. The spikes share the clock of the trial table, in the one
session there is.

## Behaviour

### beh-slopes

In [ ]:
quote("beh-slopes")


def slopes(intercept=False, produced="tp"):
    rows = []
    for (ds, prior), d in good.groupby(["dataset", "prior"]):
        y = pd.Series(rsg.produced_ms(d, produced), index=d.index)
        w = 1.0 / d.ts.map(y.groupby(d.ts).var())
        X = sm.add_constant(d.ts.astype(float)) if intercept else d.ts.astype(float)
        m = sm.WLS(y.values, X, weights=w.values).fit()
        lo, hi = m.conf_int().loc["ts"].values
        rows.append(dict(dataset=ds, prior=prior, slope=m.params["ts"], ci_lo=lo, ci_hi=hi, n=len(d)))
    return pd.DataFrame(rows)


# main reading: tp = beta ts + e (Methods, Model free analysis of behavior; Table S1 legend)
SLOPES = {"through the origin (main)": slopes(False), "with intercept": slopes(True),
          "through the origin, go_time - set_time in place of tp": slopes(False, "go"),
          "with intercept, go_time - set_time in place of tp": slopes(True, "go")}
for name, s in SLOPES.items():
    s["in_0_1"] = (s.slope > 0) & (s.slope < 1)
    s["ci_excludes_0_1"] = (s.ci_lo > 0) & (s.ci_hi < 1)
    print(f"\nreading: {name}")
    print(s.round(3).to_string(index=False))
    print(f"slopes in (0, 1): {s.in_0_1.sum()} of 8; intervals excluding 0 and 1: {s.ci_excludes_0_1.sum()} of 8")


The Methods (Analysis of behavior, "Model free analysis") and the legend of Table S1 both write the
regression as `tp = beta ts + e`, without an intercept: this is the main reading. The regression with an
intercept, and both regressions with `go_time - set_time` in place of `tp` (the data check: `tp` exceeds it
by a median of 17.9 ms), are shown as variants.

**Result: not reproduced.** Through the origin, 2 of 8 slopes lie between 0 and 1 (Eye Left Long 0.996,
Hand Left Long 0.923) and 1 of 8 intervals excludes both 0 and 1; the other six slopes are 1.013 to 1.072.
With an intercept, 6 of 8 slopes lie between 0 and 1; Eye Left Short (1.011) and Eye Right Short (1.110)
do not. With `go_time - set_time` in place of `tp`, 4 of 8 (through the origin) and 7 of 8 (with an
intercept) lie between 0 and 1. No reading has all 8 below 1.

### beh-overlap-prior

In [ ]:
quote("beh-overlap-prior")
o = good[good.ts == 800]
for prior in ["Short", "Long"]:
    x = o[o.prior == prior].tp - 800
    print(f"{prior}: n = {len(x)}, median tp - 800 = {x.median():.1f} ms, mean {x.mean():.1f} ms")
p_all = stats.ranksums(o[o.prior == "Short"].tp, o[o.prior == "Long"].tp)
print(f"rank-sum Short vs Long, pooled: z = {p_all.statistic:.2f}, p = {p_all.pvalue:.2e}")
for ds, d in o.groupby("dataset"):
    a, b = d[d.prior == "Short"].tp, d[d.prior == "Long"].tp
    r = stats.ranksums(a, b)
    print(f"  {ds}: Short median {a.median() - 800:+.1f}, Long median {b.median() - 800:+.1f} ms (n {len(a)}, {len(b)}); p = {r.pvalue:.3g}")

**Result: not reproduced.** At ts = 800 ms the median of tp - 800 is +28.3 ms for Short (n = 93) and
+47.6 ms for Long (n = 107): both are positive, and the rank-sum test between priors gives p = 0.073.
Per dataset, only Hand Left has a negative Short median (-61.6 ms), the Long median is the larger in 2
of 4 datasets, and only Hand Right differs between priors at p < 0.05 (p = 0.00099). The paper's
p < 1e-43 pools 17 sessions; this is one.

### beh-sigmoid-increments

In [ ]:
quote("beh-sigmoid-increments")
rows = []
for (ds, prior), d in good.groupby(["dataset", "prior"]):
    m = d.groupby("ts").tp.mean().values
    inc = np.diff(m)
    rows.append(dict(dataset=ds, prior=prior, increments=np.round(inc, 1).tolist(),
                     extreme=(inc[0] + inc[3]) / 2, middle=(inc[1] + inc[2]) / 2))
INC = pd.DataFrame(rows)
INC["middle_minus_extreme"] = INC.middle - INC.extreme
print(INC.round(1).to_string(index=False))
w = stats.wilcoxon(INC.middle, INC.extreme)
print(f"middle > extreme in {int((INC.middle > INC.extreme).sum())} of 8; two-sided signed-rank p = {w.pvalue:.4f}")

**Result: reproduced.** The middle increment exceeds the extreme one in 7 of 8 conditions (Hand Left
Short is the exception, -10.7 ms); two-sided signed-rank p = 0.0156, the smallest but one that 8
conditions allow. The paper's p < 1e-11 pools 17 sessions.

### beh-long-more-bias

In [ ]:
quote("beh-long-more-bias")
for name, s in SLOPES.items():
    p = s.pivot(index="dataset", columns="prior", values="slope")
    p["Long_below_Short"] = p.Long < p.Short
    print(f"\n{name}:")
    print(p.round(3).to_string())
    print(f"Long slope below Short in {int(p.Long_below_Short.sum())} of 4 datasets")

**Result: partially reproduced.** Through the origin, the equation of the Methods and of Table S1, the
Long slope is below the Short slope in 3 of 4 datasets; in Eye Right it is above (1.027 against 1.016).
With an intercept it is below in all 4 (0.845 against 1.011, 0.903 against 1.110, 0.502 against 0.612,
0.612 against 0.861), which would make the claim reproduced. With `go_time - set_time` in place of `tp`
the counts are the same, 3 of 4 and 4 of 4.

### beh-bayes-model

In [ ]:
quote("beh-bayes-model")
t = time.time()
FITS = {ds: rsg.fit_bayes(d) for ds, d in good.groupby("dataset")}
print(f"fits took {time.time() - t:.0f} s")


def model_check(d, p):
    """Observed and predicted bias and SD of tp over the 10 conditions of one dataset."""
    ob = d.groupby(["prior", "ts"]).tp.agg(["mean", "std", "size"]).reset_index()
    pr = np.array([rsg.bayes_predict(r.ts, r.prior, p)[:2] for r in ob.itertuples()])
    bias_o, bias_p = ob["mean"] - ob.ts, pr[:, 0] - ob.ts
    r2 = 1 - np.sum((bias_o - bias_p) ** 2) / np.sum((bias_o - bias_o.mean()) ** 2)
    return ob, pr, bias_o, bias_p, r2, stats.pearsonr(ob["std"], pr[:, 1])


rows, pred_rows = [], []
for ds, d in good.groupby("dataset"):
    p = FITS[ds]
    ob, pr, bias_o, bias_p, r2, rsd = model_check(d, p)
    rows.append(dict(dataset=ds, wm=p["wm"], wp=p["wp"], bShort=p["bShort"], bLong=p["bLong"],
                     bias_R2=r2, bias_r_squared=stats.pearsonr(bias_o, bias_p)[0] ** 2,
                     sd_r=rsd[0], sd_p=rsd[1], bias_range_ms=np.ptp(bias_o)))
    for r, b, q in zip(ob.itertuples(), bias_p, pr[:, 1]):
        pred_rows.append(dict(dataset=ds, prior=r.prior, ts=r.ts, n=r.size,
                              bias_obs=r.mean - r.ts, bias_model=b, sd_obs=r.std, sd_model=q))
BAYES = pd.DataFrame(rows)
print(BAYES.round(3).to_string(index=False))
print(f"bias R2 >= 0.8 in {int((BAYES.bias_R2 >= 0.8).sum())} of 4 (squared correlation >= 0.8 in {int((BAYES.bias_r_squared >= 0.8).sum())}); "
      f"SD correlation positive in {int((BAYES.sd_r > 0).sum())} of 4")
print(pd.DataFrame(pred_rows).round(1).to_string(index=False))


In [ ]:
# Table S3 of the paper (supplement): wm and wp of monkey H, mean +- SD over sessions fit one by one
TABLE_S3 = {"Hand Left": (0.085, 0.025, 0.086, 0.020), "Hand Right": (0.062, 0.026, 0.084, 0.021),
            "Eye Left": (0.043, 0.016, 0.086, 0.022), "Eye Right": (0.027, 0.009, 0.077, 0.033)}
t = time.time()
rows = []
for ds, d in good.groupby("dataset"):
    p = FITS[ds]
    wm3, wm3_sd, wp3, wp3_sd = TABLE_S3[ds]
    relaxed = rsg.fit_bayes(d, w_min=1e-4)
    fixed = rsg.fit_bayes(d, wm_fixed=wm3)
    *_, r2_fixed, rsd_fixed = model_check(d, fixed)
    rows.append(dict(dataset=ds, wm=p["wm"], wm_S3=f"{wm3} +- {wm3_sd}", wm_z=(p["wm"] - wm3) / wm3_sd,
                     wp=p["wp"], wp_S3=f"{wp3} +- {wp3_sd}", wp_z=(p["wp"] - wp3) / wp3_sd,
                     wm_bound_1e4=relaxed["wm"], dLL_bound_1e4=relaxed["loglik"] - p["loglik"],
                     wp_at_wm_S3=fixed["wp"], dLL_at_wm_S3=fixed["loglik"] - p["loglik"],
                     bias_R2_at_wm_S3=r2_fixed, sd_r_at_wm_S3=rsd_fixed[0]))
S3 = pd.DataFrame(rows)
print("fitted parameters against Table S3 (z = (fit - mean) / SD over sessions), the fit with the lower bound on the")
print("noise weights relaxed from 0.01 to 1e-4, and the fit with wm held at Table S3's mean:")
print(S3.round(3).to_string(index=False))
print(f"with wm at Table S3's mean: bias R2 >= 0.8 in {int((S3.bias_R2_at_wm_S3 >= 0.8).sum())} of 4, "
      f"SD correlation positive in {int((S3.sd_r_at_wm_S3 > 0).sum())} of 4")
d = good[good.dataset == "Eye Right"]
prof = []
for wm in [0.003, 0.006, 0.01, 0.013, 0.02, 0.027, 0.04, 0.06]:
    f = rsg.fit_bayes(d, w_min=1e-4, wm_fixed=wm)
    prof.append(dict(wm=wm, loglik_minus_best=f["loglik"] - FITS["Eye Right"]["loglik"], bias_R2=model_check(d, f)[4]))
print("\nEye Right: profile log-likelihood over wm (other parameters refit), relative to the best fit")
print(pd.DataFrame(prof).round(3).to_string(index=False))
print(f"took {time.time() - t:.0f} s")


"R2 of the observed mean bias by the model's predicted bias" is read as 1 - SS_res / SS_tot of the
observed biases around the predictions; the squared Pearson correlation is shown as the second reading.
The second cell compares the fitted wm and wp with Table S3 of the paper (monkey H, mean and SD over
sessions fit one by one), refits with the lower bound on the noise weights relaxed, refits with wm held at
Table S3's mean, and profiles the likelihood of the Eye Right fit over wm.

**Result: partially reproduced.** The correlation of observed and predicted SD of tp is positive in
all 4 datasets (0.807, 0.909, 0.749, 0.277; p = 0.44 for Hand Right), but the bias R2 reaches 0.8 in 2
of 4: Hand Left 0.978 and Hand Right 0.865, against Eye Left 0.670 and Eye Right 0.122. The squared
correlation gives the same count (0.678, 0.129, 0.978, 0.869). The observed Eye biases span 108 and 80 ms,
against 265 and 166 ms for Hand. Against Table S3, the fitted wm lie 1.1 and 1.5 SD below the session mean
for Eye Left (0.025) and Eye Right (0.013), 1.3 SD above for Hand Left (0.116) and 0.4 SD above for Hand
Right (0.073); the fitted wp (0.090 to 0.106) lie 0.2 to 1.0 SD above. The fits are interior: with the
lower bound relaxed from 0.01 to 1e-4 they do not move. With wm held at Table S3's mean, the bias R2 is
0.581, 0.126, 0.929 and 0.830, again 2 of 4, so the low Eye wm do not decide the outcome. For Eye Right
the profile log-likelihood is within 0.46 of its best from wm 0.003 to 0.02, so wm is poorly determined
there, and the bias R2 stays at or below 0.146 over the whole profile.

## Single neurons

### glm-prior-ts

Counts in the 80 ms before each of the Short prior's ts values, with attrition. For ts dependence the
criterion gives each prior "that prior's 5 time indicators". Reading (a): for the Long prior, windows
ending at the Long prior's ts values (800 to 1200 ms), as the Methods' "Since values of ts were
different between the priors, we used two distinct GLMs" suggests; reading (b): the same five windows
(480 to 800 ms) in both priors.

In [ ]:
quote("glm-prior-ts")


def window_counts(d, ends):
    D, C = [], []
    for e in ends:
        c = rsg.bin_counts(spikes, d.ready_time.values, e - 80, e, bin_ms=80)[:, 0, :]
        ok = d.ts.values >= e
        D.append(pd.DataFrame({"time": e, "long": (d.prior == "Long").astype(float).values,
                               "hand": (d.effector == "Hand").astype(float).values,
                               "left": (d.direction == "Left").astype(float).values})[ok])
        C.append(c[ok])
    return pd.concat(D, ignore_index=True), np.vstack(C)


def bic(y, X):
    return sm.GLM(y, X, family=sm.families.Poisson()).fit().bic_llf


def binom_ci(k, n):
    lo = stats.beta.ppf(0.025, k, n - k + 1) if k > 0 else 0.0
    hi = stats.beta.ppf(0.975, k + 1, n - k) if k < n else 1.0
    return 100 * lo, 100 * hi


D, C = window_counts(good, rsg.SHORT_TS)
T = pd.get_dummies(D.time, prefix="t").astype(float)
full = pd.concat([T, D[["long", "hand", "left"]]], axis=1)
red = pd.concat([T, D[["hand", "left"]]], axis=1)
prior_dep = np.array([bic(C[:, k], full) < bic(C[:, k], red) for k in range(C.shape[1])])
print(f"observations per unit (trial x window, with attrition): {len(D)}")
nU = len(prior_dep)
GLM = {}
for reading in ["a", "b"]:
    ts_dep = np.zeros(nU, bool)
    for prior in ["Short", "Long"]:
        d = good[good.prior == prior]
        ends = rsg.PRIOR_TS[prior] if reading == "a" else rsg.SHORT_TS
        D2, C2 = window_counts(d, ends)
        T2 = pd.get_dummies(D2.time, prefix="t").astype(float)
        X3 = pd.concat([T2, D2[["hand", "left"]]], axis=1)
        X4 = sm.add_constant(D2[["hand", "left"]])
        ts_dep |= np.array([bic(C2[:, k], X3) < bic(C2[:, k], X4) for k in range(nU)])
    GLM[reading] = ts_dep
e_prior, e_ts = expected("glm-prior-ts")
fp = 100 * prior_dep.mean()
print(f"prior-dependent: {prior_dep.sum()} of {nU} = {fp:.1f}% (95% CI {binom_ci(prior_dep.sum(), nU)[0]:.1f} to {binom_ci(prior_dep.sum(), nU)[1]:.1f}); "
      f"expected {e_prior['value']} +- {e_prior['tolerance']}: within {within(fp, e_prior)}")
for reading, ts_dep in GLM.items():
    f = 100 * ts_dep.mean()
    lo, hi = binom_ci(ts_dep.sum(), nU)
    print(f"ts-dependent, reading ({reading}): {ts_dep.sum()} of {nU} = {f:.1f}% (95% CI {lo:.1f} to {hi:.1f}); "
          f"expected {e_ts['value']} +- {e_ts['tolerance']}: within {within(f, e_ts)}")
print("both prior- and ts-dependent (a):", int((prior_dep & GLM['a']).sum()), "| neither:", int((~prior_dep & ~GLM['a']).sum()))

**Result: not reproduced.** 38 of 48 units (79.2%, 95% CI 65.0 to 89.5) are prior-dependent, against
65 +- 6.5%, and 22 of 48 (45.8%, 95% CI 31.4 to 60.8) are ts-dependent under reading (a), 24 of 48
(50.0%, 35.2 to 64.8) under reading (b), against 27 +- 2.7%. Both fractions are above the paper's under
both readings. The tolerances are narrower than the binomial interval of a fraction of 48 units: the
prior fraction's interval reaches 65, while both ts fractions' intervals exclude 27.

### dfr-overlap-support

In [ ]:
quote("dfr-overlap-support")
EDGES = np.arange(-320, 1441, 160)
rng = np.random.default_rng(2019)
o800 = {ds: good[(good.dataset == ds) & (good.ts == 800)] for ds in good.dataset.unique()}
counts800 = {ds: rsg.bin_counts(spikes, d.ready_time.values, -320, 1440, bin_ms=160) / 0.16 for ds, d in o800.items()}


def rms_curve(shuffle=False):
    out = []
    for ds, d in o800.items():
        lab = (d.prior == "Long").values
        if shuffle:
            lab = rng.permutation(lab)
        diff = counts800[ds][lab].mean(0) - counts800[ds][~lab].mean(0)
        out.append(np.sqrt((diff ** 2).mean(axis=1)))
    return np.array(out)


per_ds = rms_curve()
obs = per_ds.mean(0)
null = np.array([rms_curve(True).mean(0).max() for _ in range(200)])
print("trials at ts = 800 (Short, Long):", {ds: [int((d.prior == p).sum()) for p in ["Short", "Long"]] for ds, d in o800.items()})
tab = pd.DataFrame(per_ds.T, columns=list(o800), index=[f"{a} to {b}" for a, b in zip(EDGES[:-1], EDGES[1:])])
tab["mean"] = obs
print("RMS over units of the Long - Short rate difference (spikes/s), by 160 ms bin after Ready:")
print(tab.round(2).to_string())
i = int(np.argmax(obs))
print(f"largest mean RMS {obs[i]:.2f} spikes/s in bin {EDGES[i]} to {EDGES[i + 1]} ms; "
      f"95th percentile of the shuffled maximum {np.percentile(null, 95):.2f} (range {null.min():.2f} to {null.max():.2f})")
print("largest bin within 480 to 800 ms:", EDGES[i] >= 480 and EDGES[i + 1] <= 800)

**Result: reproduced.** The mean RMS difference is largest in the 480 to 640 ms bin (6.86 spikes/s),
above the 95th percentile of the shuffled maximum (3.15; the largest of 200 shuffles is 3.46). The
next largest are the bin after Set, 800 to 960 ms (6.07), and 320 to 480 ms (5.97). Per dataset, the
largest bin lies within 480 to 800 ms in 3 of 4 (Eye Right 9.14, Hand Left 4.88, Hand Right 4.39);
in Eye Left it is 800 to 960 ms (9.73).

## Population trajectories

Per dataset: estimation-epoch trajectories (Ready to the prior's longest ts, attrition) and
production-epoch trajectories (Set to the shortest tp of each ts), PCA per epoch on both priors, and the
encoding axis of each prior. Each dataset uses its own Bayesian fit (beh-bayes-model) for te(t).

In [ ]:
GEO = {ds: rsg.dataset_geometry(spikes, good[good.dataset == ds], FITS[ds]["wm"]) for ds in sorted(good.dataset.unique())}
W = pd.concat([g["prior"][p]["per_ts"].assign(dataset=ds, prior=p) for ds, g in GEO.items() for p in ["Short", "Long"]], ignore_index=True)
W["past_go_ms"] = W.duration_ms - W.min_go_set_ms
print("production windows (Set to the shortest tp, in 20 ms bins) against the shortest go_time - set_time of each condition (ms):")
print(W[["dataset", "prior", "ts", "n_trials", "duration_ms", "min_go_set_ms", "past_go_ms"]].round(1).to_string(index=False))
print(f"windows ending after the earliest Go of their condition: {int((W.past_go_ms > 0).sum())} of {len(W)}; "
      f"by up to {W.past_go_ms.max():.1f} ms (median {W.past_go_ms.median():.1f})")


### pca-dimensionality

In [ ]:
quote("pca-dimensionality")
rows = []
for ds, g in GEO.items():
    rows.append(dict(dataset=ds, k_estimation=g["k_est"], cum_var_est=np.round(np.cumsum(g["ev_est"])[:5], 3).tolist(),
                     k_production=g["k_prod"], cum_var_prod=np.round(np.cumsum(g["ev_prod"])[:12], 3).tolist()))
PCA = pd.DataFrame(rows)
print(PCA.to_string(index=False))
n_in = int(PCA.k_estimation.between(3, 4).sum() + PCA.k_production.between(5, 11).sum())
print(f"estimation in 3 to 4: {int(PCA.k_estimation.between(3, 4).sum())} of 4; production in 5 to 11: "
      f"{int(PCA.k_production.between(5, 11).sum())} of 4; total {n_in} of 8")
for ds, g in GEO.items():
    print(f"{ds}: variance of 3 and 4 estimation PCs {np.cumsum(g['ev_est'])[2]:.3f}, {np.cumsum(g['ev_est'])[3]:.3f}")

**Result: not reproduced.** In the estimation epoch, 2 components reach 75% of the variance in all 4
datasets (0.80 to 0.86 with 2), so none is in 3 to 4; in the production epoch 2, 2, 4 and 5 components
do, so 1 of 4 is in 5 to 11. In all, 1 of 8 values is in range. The counts depend on the smoothing
(robustness table below): with a kernel SD of 20 ms, 2 of 8 are in range, and with SD 17 ms (a full width
at half maximum of 40 ms) 3 of 8, both still not reproduced; without smoothing 5 of 8 are, which would be
partially reproduced. The paper's PCA uses the units of all sessions of an animal (536 in monkey H); here
there are 48 units of one session. Both the reading of the kernel width and the pooled population are
asked of a reader (questions/sohn-2019.yaml).

### encoding-r2

In [ ]:
quote("encoding-r2")
rng = np.random.default_rng(4)
rows = []
for ds, g in GEO.items():
    for prior in ["Short", "Long"]:
        q = g["prior"][prior]
        a, b, fit_te, r2_te = rsg.wls(q["xu"], q["te"], q["w"])
        a2, b2, fit_ts, r2_ts = rsg.wls(q["xu"], q["t"], q["w"])
        sel = (q["ends"] >= q["t"][0]) & (q["ends"] <= q["t"][-1])
        V = rng.normal(size=(1000, q["X"].shape[1]))
        V /= np.linalg.norm(V, axis=1, keepdims=True)
        r2_rand = np.array([rsg.wls(q["X"][sel] @ v, q["te"], q["w"])[3] for v in V])
        rows.append(dict(dataset=ds, prior=prior, n_points=len(q["t"]), k=q["X"].shape[1], R2_te=r2_te, R2_ts=r2_ts,
                         rand_median=np.median(r2_rand), rand_95=np.percentile(r2_rand, 95),
                         frac_rand_ge=np.mean(r2_rand >= r2_te)))
ENC = pd.DataFrame(rows)
print(ENC.round(4).to_string(index=False))
eS, eL = expected("encoding-r2")
hl = ENC[ENC.dataset == "Hand Left"].set_index("prior").R2_te
mean = ENC.groupby("prior").R2_te.mean()
for name, v in [("(a) Hand Left", hl), ("(b) mean of 4 datasets", mean)]:
    print(f"{name}: Short {v['Short']:.4f} (within {within(v['Short'], eS)}), Long {v['Long']:.4f} (within {within(v['Long'], eL)})")

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(14, 6))
for j, (ds, g) in enumerate(GEO.items()):
    for i, prior in enumerate(["Short", "Long"]):
        q = g["prior"][prior]
        a, b, fit, r2 = rsg.wls(q["xu"], q["te"], q["w"])
        ax = axes[i, j]
        ax.plot(q["t"], q["xu"], "o", ms=3, color="C3" if prior == "Short" else "C0")
        ax.plot(q["t"], fit, "k-", lw=1)
        ax.set_title(f"{ds} {prior}: R2 {r2:.3f}", fontsize=9)
        ax.set_xlabel("time after Ready (ms)")
        print(f"{ds} {prior}: Xu from {q['xu'].min():.2f} to {q['xu'].max():.2f} spikes/s (all points shown)")
axes[0, 0].set_ylabel("Xu (spikes/s)")
plt.tight_layout()
plt.show()

**Result: reproduced.** Under reading (a), Hand Left, R2 is 0.996 for Short and 0.977 for Long; under
reading (b), the mean of the 4 datasets, 0.959 and 0.982. All four values are within tolerance. The
reference is weak in this space of 2 components: random unit vectors give median R2 of 0.69 to 0.99,
from 1.2% to 44.2% of them reach the R2 of u, and ts itself explains Xu almost as well as te (R2 0.898
to 0.990). With the narrower kernels of the robustness
table, Hand Left gives 0.982 and 0.916 (SD 20 ms) and 0.994 and 0.937 (SD 17 ms), still within tolerance.


### encoding-bayes-beats-linear

In [ ]:
quote("encoding-bayes-beats-linear")
rows = []
for ds, g in GEO.items():
    for prior in ["Short", "Long"]:
        q = g["prior"][prior]
        _, _, f_te, _ = rsg.wls(q["xu"], q["te"], q["w"])
        _, _, f_ts, _ = rsg.wls(q["xu"], q["t"], q["w"])
        w = q["w"] / q["w"].sum()
        rows.append(dict(dataset=ds, prior=prior,
                         rmse_bayes=np.sqrt(np.mean((q["xu"] - f_te) ** 2)), rmse_linear=np.sqrt(np.mean((q["xu"] - f_ts) ** 2)),
                         wrmse_bayes=np.sqrt(np.sum(w * (q["xu"] - f_te) ** 2)), wrmse_linear=np.sqrt(np.sum(w * (q["xu"] - f_ts) ** 2))))
RM = pd.DataFrame(rows)
RM["delta"] = RM.rmse_bayes - RM.rmse_linear
RM["w_delta"] = RM.wrmse_bayes - RM.wrmse_linear
print(RM.round(3).to_string(index=False))
for prior in ["Short", "Long"]:
    r = RM[RM.prior == prior]
    print(f"{prior}: Bayesian RMSE lower in {int((r.delta < 0).sum())} of 4 (weighted RMSE: {int((r.w_delta < 0).sum())} of 4); "
          f"signed-rank p = {stats.wilcoxon(r.delta).pvalue:.3f}")

**Result: partially reproduced.** The Bayesian fit has the lower RMSE in all 4 datasets for Short
(differences -0.04 to -0.47 spikes/s) and in 3 of 4 for Long; for Hand Left Long the linear fit is
better (0.481 against 0.583). The weighted RMSE gives the same counts. The signed-rank p over 4
datasets is 0.125 for Short and 0.25 for Long; 4 datasets cannot give the paper's 0.008. With the narrower kernels of the
robustness table the Bayesian fit is better in 3 of 4 datasets for each prior (Hand Right Short +0.055
and +0.064, Hand Left Long +0.113 and +0.082), which would be not reproduced; the claims file fixes an SD
of 40 ms, and the reading of the kernel width is asked of a reader.


### short-in-long-control

Long-prior states from 480 to 800 ms after Ready, with an axis from the state at 480 to that at 800 ms.
The main reading builds the axis as u is built (the mean of the difference vectors; the Methods'
"applied the same projection analysis"); the single difference X(800) - X(480) is the second reading.

In [ ]:
quote("short-in-long-control")
rows = []
for ds, g in GEO.items():
    q = g["prior"]["Long"]
    t = np.arange(480, 801, 20)
    idx = [int(np.flatnonzero(q["ends"] == x)[0]) for x in t]
    X = q["X"][idx]
    te_short = rsg.te_mean(t.astype(float), "Short", FITS[ds]["wm"])
    w = q["n"][idx]
    for axis_name in ["mean of differences", "single difference"]:
        if axis_name == "mean of differences":
            u = rsg.encoding_axis(q["X"], q["ends"], 480, 800)
        else:
            u = (X[-1] - X[0]) / np.linalg.norm(X[-1] - X[0])
        xu = X @ u
        _, _, f_te, r2b = rsg.wls(xu, te_short, w)
        _, _, f_ts, r2l = rsg.wls(xu, t.astype(float), w)
        rows.append(dict(dataset=ds, axis=axis_name, rmse_bayes=np.sqrt(np.mean((xu - f_te) ** 2)),
                         rmse_linear=np.sqrt(np.mean((xu - f_ts) ** 2)), R2_te=r2b, R2_ts=r2l))
SIL = pd.DataFrame(rows)
SIL["delta"] = SIL.rmse_bayes - SIL.rmse_linear
print(SIL.round(4).to_string(index=False))
for axis_name, r in SIL.groupby("axis"):
    print(f"{axis_name}: delta RMSE (Bayesian - linear) {np.round(r.delta.values, 3).tolist()}; "
          f"two-sided signed-rank p = {stats.wilcoxon(r.delta).pvalue:.3f}")
sp_delta = RM[RM.prior == "Short"].delta.values
print("for comparison, Short prior delta RMSE:", np.round(sp_delta, 3).tolist())
# the paper's two-way repeated-measures ANOVA: prior condition (Short, Short In Long) x model (Bayesian, linear)
from statsmodels.stats.anova import AnovaRM
for axis_name, r in SIL.groupby("axis"):
    long_form = []
    for ds in r.dataset:
        s_row = RM[(RM.dataset == ds) & (RM.prior == "Short")].iloc[0]
        l_row = r[r.dataset == ds].iloc[0]
        long_form += [dict(dataset=ds, condition="Short", model="Bayesian", rmse=s_row.rmse_bayes),
                      dict(dataset=ds, condition="Short", model="linear", rmse=s_row.rmse_linear),
                      dict(dataset=ds, condition="Short In Long", model="Bayesian", rmse=l_row.rmse_bayes),
                      dict(dataset=ds, condition="Short In Long", model="linear", rmse=l_row.rmse_linear)]
    a = AnovaRM(pd.DataFrame(long_form), "rmse", "dataset", within=["condition", "model"]).fit().anova_table
    row = a.loc["condition:model"]
    print(f"{axis_name}: interaction condition x model F({row['Num DF']:.0f},{row['Den DF']:.0f}) = {row['F Value']:.2f}, p = {row['Pr > F']:.3f}")


**Result: reproduced.** The signed-rank test of the RMSE difference over the 4 datasets gives
p = 0.25 (single-difference axis: p = 0.125), so the paper's test finds no difference; with 4
datasets it cannot reach p < 0.05. The effect sizes do not show an absence: the Bayesian model fits
better in 3 of 4 datasets (differences -0.21, -0.12 and -0.11 spikes/s; Hand Left +0.05), of a size
like the Short prior's own (-0.04 to -0.47). The paper's two-way
repeated-measures ANOVA (Short or Short In Long, by model) gives an interaction of F(1,3) = 1.70,
p = 0.284 (single-difference axis: F(1,3) = 2.32, p = 0.225), against the paper's F(1,7) = 161.43,
p < 1e-5.


### neural-long-more-bias

In [ ]:
quote("neural-long-more-bias")
rows = []
for ds, g in GEO.items():
    r = {"dataset": ds}
    for prior in ["Short", "Long"]:
        q = g["prior"][prior]
        r[prior] = rsg.wls(q["xu"], q["t"], q["w"])[1]
    rows.append(r)
NS = pd.DataFrame(rows)
NS["Long_below_Short"] = NS.Long < NS.Short
NS["ratio_Long_Short"] = NS.Long / NS.Short
print("slope of Xu on ts (spikes/s per ms):")
print(NS.round(4).to_string(index=False))
print(f"Long below Short in {int(NS.Long_below_Short.sum())} of 4; signed-rank p (Short - Long) = {stats.wilcoxon(NS.Short - NS.Long).pvalue:.3f}")

**Result: partially reproduced.** The slope of Xu on ts is lower for Long in 3 of 4 datasets
(Long/Short ratios 0.54, 0.62 and 0.36); in Hand Right the Long slope is slightly higher (0.0486
against 0.0470). With the outlier trials included the count is the same; without smoothing the Long
slope is lower in all 4 (robustness table below). With the narrower kernels the count is also 3 of 4.


### Robustness of the trajectory measures

The same measures with the outlier-flagged completed trials included; without smoothing; with narrower
kernels, SD 20 ms and SD 17 ms (a full width at half maximum of 40 ms), two other readings of the
Methods' "Gaussian smoothing kernel width: 40 ms", since the same Methods write "a standard deviation of
25 ms" where they mean an SD; and with each production window ending at the shortest `go_time - set_time`
of its condition, so that no window runs past Go (with windows ending at the shortest `tp`, 36 of 40 do,
by up to 121.5 ms; table above). The second table gives what each variant yields on the criteria.

In [ ]:
all_completed = rsg.label_trials(sess["trials"])
all_completed = all_completed[all_completed.set_time.notna() & all_completed.go_time.notna() & (all_completed.tp > 0)].reset_index(drop=True)
variants = [("main", good, True, rsg.SMOOTH_SD, "tp"), ("with outlier trials", all_completed, True, rsg.SMOOTH_SD, "tp"),
            ("unsmoothed", good, False, rsg.SMOOTH_SD, "tp"), ("SD 20 ms", good, True, 20, "tp"),
            ("SD 17 ms (FWHM 40 ms)", good, True, 40 / 2.3548, "tp"),
            ("windows end at go_time - set_time", good, True, rsg.SMOOTH_SD, "go")]
rows, GEOV = [], {}
for name, trials, smooth, sd, end in variants:
    for ds in sorted(good.dataset.unique()):
        g = GEO[ds] if name == "main" else rsg.dataset_geometry(spikes, trials[trials.dataset == ds], FITS[ds]["wm"], smooth=smooth, sd=sd, end=end)
        GEOV[(name, ds)] = g
        r = dict(variant=name, dataset=ds, k_est=g["k_est"], k_prod=g["k_prod"])
        for prior in ["Short", "Long"]:
            q = g["prior"][prior]
            _, _, f_te, r2 = rsg.wls(q["xu"], q["te"], q["w"])
            _, slope, f_ts, _ = rsg.wls(q["xu"], q["t"], q["w"])
            r[f"R2_{prior}"] = r2
            r[f"dRMSE_{prior}"] = np.sqrt(np.mean((q["xu"] - f_te) ** 2)) - np.sqrt(np.mean((q["xu"] - f_ts) ** 2))
            r[f"slope_{prior}"] = slope
        rows.append(r)
ROB = pd.DataFrame(rows)
print(ROB.round(4).to_string(index=False))
print("\nwhat each variant gives on the criteria (speed r pooled over datasets, 20 points per prior):")
summ = []
for name, r in ROB.groupby("variant", sort=False):
    sp = pd.concat([GEOV[(name, ds)]["prior"][p]["per_ts"].assign(prior=p) for ds in r.dataset for p in ["Short", "Long"]])
    hl = r[r.dataset == "Hand Left"].iloc[0]
    summ.append(dict(variant=name,
                     pca_in_range=int(r.k_est.between(3, 4).sum() + r.k_prod.between(5, 11).sum()),
                     R2_HandLeft=f"{hl.R2_Short:.3f}, {hl.R2_Long:.3f}",
                     R2_mean=f"{r.R2_Short.mean():.3f}, {r.R2_Long.mean():.3f}",
                     bayes_better_Short=int((r.dRMSE_Short < 0).sum()), bayes_better_Long=int((r.dRMSE_Long < 0).sum()),
                     long_below_short=int((r.slope_Long < r.slope_Short).sum()),
                     r_speed_xu=", ".join(f"{stats.pearsonr(sp[sp.prior == p].xu_set, sp[sp.prior == p].speed)[0]:.3f}" for p in ["Short", "Long"]),
                     r_speed_tp=", ".join(f"{stats.pearsonr(sp[sp.prior == p].speed, sp[sp.prior == p].mean_tp)[0]:.3f}" for p in ["Short", "Long"])))
print(pd.DataFrame(summ).to_string(index=False))


## Speed in the production epoch

### speed-vs-projection

In [ ]:
quote("speed-vs-projection")
SP = pd.concat([g["prior"][p]["per_ts"].assign(dataset=ds, prior=p) for ds, g in GEO.items() for p in ["Short", "Long"]], ignore_index=True)
# the Methods: "average Euclidean distance ... between neural states associated with successive bins (20 ms), divided by
# the duration separating Set and the time of Go". Variant (i) divides the mean step by the mean Set to Go duration, as
# written; variant (ii) divides the path length (the summed steps) by that duration.
SP["speed_div_duration"] = SP.speed / (SP.mean_go_set_ms / 1000.0)
SP["path_per_s"] = SP.path / (SP.mean_go_set_ms / 1000.0)
print(SP[["dataset", "prior", "ts", "n_trials", "duration_ms", "xu_set", "speed", "speed_div_duration", "path_per_s", "mean_tp"]].round(3).to_string(index=False))
eS, eL = expected("speed-vs-projection")
for prior, e in [("Short", eS), ("Long", eL)]:
    q = SP[SP.prior == prior]
    r = stats.pearsonr(q.xu_set, q.speed)
    rv = stats.pearsonr(q.xu_set, q.speed_div_duration)
    rp = stats.pearsonr(q.xu_set, q.path_per_s)
    within_ds = [stats.pearsonr(z.xu_set, z.speed)[0] for _, z in q.groupby("dataset")]
    print(f"{prior}: r = {r[0]:.3f}, p = {r[1]:.2g} (expected {e['value']} +- {e['tolerance']}: within {within(r[0], e)}); "
          f"variant (i) {rv[0]:.3f}, p = {rv[1]:.2g}; variant (ii) {rp[0]:.3f}, p = {rp[1]:.2g}; within datasets {np.round(within_ds, 3).tolist()}")


**Result: partially reproduced.** Larger projections at Set go with slower trajectories in both
priors: r = -0.932 (p = 2.3e-09) for Short and -0.874 (p = 4.8e-07) for Long, both negative and
stronger than the paper's -0.74 and -0.51, outside tolerance. Within datasets r is -0.919 to -0.995
(Short) and -0.669 to -0.956 (Long). Dividing the mean step by the mean Set to Go duration, as the
Methods word it, gives -0.922 and -0.924; dividing the path length by that duration gives -0.933 and
-0.875. With windows ending before Go r is -0.921 and -0.838; with a kernel SD of 20 ms -0.663 and
-0.618, and with SD 17 ms -0.554 and -0.465, the last within tolerance for Long, which is also partially
reproduced.

### initial-speed-biased

In [ ]:
quote("initial-speed-biased")
rows = []
for ds, g in GEO.items():
    d = good[good.dataset == ds]
    mu, comps = g["prod_pca"]
    for prior in ["Short", "Long"]:
        ini = rsg.production_rates(spikes, d, prior, stop_ms=400)
        tsv = rsg.PRIOR_TS[prior].astype(float)
        sv = np.array([rsg.step_speed(rsg.project(ini[ts], mu, comps)) for ts in rsg.PRIOR_TS[prior]])
        unb = sv[2] * tsv[2] / tsv
        n = [int(((d.prior == prior) & (d.ts == ts) & (d.tp >= 400)).sum()) for ts in rsg.PRIOR_TS[prior]]
        rows.append(dict(dataset=ds, prior=prior, speeds=np.round(sv, 2).tolist(), n_trials=n,
                         slope_measured=stats.linregress(tsv, sv).slope, slope_unbiased=stats.linregress(tsv, unb).slope))
INI = pd.DataFrame(rows)
INI["shallower"] = INI.slope_measured > INI.slope_unbiased
print(INI.to_string(index=False))
w = stats.wilcoxon(INI.slope_measured - INI.slope_unbiased)
print(f"measured slope shallower than unbiased in {int(INI.shallower.sum())} of 8; two-sided signed-rank p = {w.pvalue:.4f}")
print("trials dropped for tp < 400 ms:", int((good.tp < 400).sum()))

**Result: not reproduced.** The measured slope is shallower than the unbiased one in 7 of 8
dataset-priors (Eye Right Long is the exception), but the two-sided signed-rank test gives p = 0.055,
above 0.05. No trial had tp below 400 ms.

### speed-predicts-tp

In [ ]:
quote("speed-predicts-tp")
eS, eL = expected("speed-predicts-tp")
for prior, e in [("Short", eS), ("Long", eL)]:
    q = SP[SP.prior == prior]
    r = stats.pearsonr(q.speed, q.mean_tp)
    rv = stats.pearsonr(q.speed_div_duration, q.mean_tp)
    rp = stats.pearsonr(q.path_per_s, q.mean_tp)
    within_ds = [stats.pearsonr(z.speed, z.mean_tp)[0] for _, z in q.groupby("dataset")]
    print(f"{prior}: r = {r[0]:.3f}, p = {r[1]:.2g} (expected {e['value']} +- {e['tolerance']}: within {within(r[0], e)}); "
          f"variant (i) {rv[0]:.3f}; variant (ii) {rp[0]:.3f}; within datasets {np.round(within_ds, 3).tolist()}")
print("\nvariant: trial-by-trial correlation of single-trial speed with tp, per prior, effector and direction;")
print("each trial's window ends at its tp (main) or at its go_time - set_time (go)")
rows = []
for ds, g in GEO.items():
    for prior in ["Short", "Long"]:
        d = good[(good.dataset == ds) & (good.prior == prior)]
        r = dict(dataset=ds, prior=prior, n=len(d))
        for end in ["tp", "go"]:
            s = rsg.single_trial_speed(spikes, d, g["prod_pca"], end=end)
            c = stats.pearsonr(s, d.tp)
            r[f"r_{end}"], r[f"p_{end}"] = c[0], c[1]
        rows.append(r)
TT = pd.DataFrame(rows)
print(TT.round(4).to_string(index=False))


**Result: partially reproduced.** Faster trajectories go with shorter mean tp: r = -0.719
(p = 0.00035) for Short and -0.744 (p = 0.00017) for Long, both negative with p < 0.05 and both
outside the tolerance around -0.58 and -0.52. Dividing the mean step by the mean Set to Go duration
gives -0.853 and -0.855; dividing the path length by it gives -0.613 and -0.752, the first within
tolerance, which is also partially reproduced. With windows ending before Go r is -0.693 and -0.693;
with a kernel SD of 20 ms -0.840 and -0.789, and with SD 17 ms -0.796 and -0.691. Trial by trial,
single-trial speed correlates negatively with tp in all 8 groups (r -0.15 to -0.71), with p < 0.05 in 7,
whether each trial's window ends at its tp or at its Go.

## Alternative mechanisms

### alt-speed-unmodulated

In [ ]:
quote("alt-speed-unmodulated")
rng = np.random.default_rng(5)


def support_speed_slope(d, prior):
    r, ends, n = rsg.estimation_rates(spikes, d, rsg.PRIOR_TS[prior][-1])
    v = np.linalg.norm(np.diff(r, axis=0), axis=1)
    te = ends[1:]
    sel = (te > rsg.PRIOR_TS[prior][0]) & (te <= rsg.PRIOR_TS[prior][-1])
    return stats.linregress(te[sel], v[sel]).slope, v[sel].mean()


rows = []
for ds in sorted(good.dataset.unique()):
    for prior in ["Short", "Long"]:
        d = good[(good.dataset == ds) & (good.prior == prior)]
        slope, mean_speed = support_speed_slope(d, prior)
        boot = []
        for _ in range(200):
            idx = np.concatenate([rng.choice(np.flatnonzero(d.ts.values == x), (d.ts == x).sum()) for x in rsg.PRIOR_TS[prior]])
            boot.append(support_speed_slope(d.iloc[idx], prior)[0])
        lo, hi = np.percentile(boot, [2.5, 97.5])
        span = rsg.PRIOR_TS[prior][-1] - rsg.PRIOR_TS[prior][0]
        # the same trajectory from the longest-ts trials alone: every bin averages the same trials
        dl = d[d.ts == rsg.PRIOR_TS[prior][-1]]
        slope_c, mean_c = support_speed_slope(dl, prior)
        rows.append(dict(dataset=ds, prior=prior, slope=slope, ci_lo=lo, ci_hi=hi, boot_below_pct=100 * np.mean(np.array(boot) < slope),
                         basic_lo=2 * slope - hi, basic_hi=2 * slope - lo,
                         mean_speed=mean_speed, change_over_support_pct=100 * slope * span / mean_speed,
                         n_longest=len(dl), slope_longest_only=slope_c, change_longest_only_pct=100 * slope_c * span / mean_c))
ALT1 = pd.DataFrame(rows)
print("slope of the step distance (spikes/s per 20 ms step) on time (ms) over the support, full 48-unit space:")
print(ALT1.round(5).to_string(index=False))
for prior in ["Short", "Long"]:
    s = ALT1[ALT1.prior == prior].slope
    c = ALT1[ALT1.prior == prior].slope_longest_only
    print(f"{prior}: two-sided signed-rank of 4 slopes against 0, p = {stats.wilcoxon(s).pvalue:.3f}; positive in {int((s > 0).sum())} of 4; "
          f"longest-ts trials only: p = {stats.wilcoxon(c).pvalue:.3f}, positive in {int((c > 0).sum())} of 4")
print("percentile intervals excluding 0:", int(((ALT1.ci_lo > 0) | (ALT1.ci_hi < 0)).sum()), "of 8; basic intervals excluding 0:",
      int(((ALT1.basic_lo > 0) | (ALT1.basic_hi < 0)).sum()), "of 8; bootstrap slopes below the point estimate: median",
      f"{ALT1.boot_below_pct.median():.1f}% ({ALT1.boot_below_pct.min():.1f}% to {ALT1.boot_below_pct.max():.1f}%)")

**Result: reproduced.** The signed-rank tests of the 4 slopes against zero give p = 0.25 (Short) and
0.125 (Long), so the paper's test finds no modulation; with 4 datasets it cannot reach p < 0.05. The
slope is positive in 7 of 8 dataset-priors (the step distance changes by -13% to +56% over the support),
but this follows from attrition: later bins average fewer trials, which adds noise to their steps. Built
from the longest-ts trials alone, so that every bin averages the same 18 to 31 trials, the slopes are
negative in 7 of 8 (changes of -42% to +11%; signed-rank p = 0.625 and 0.125). The percentile bootstrap
intervals exclude zero in 6 of 8, but only 3.5% to 27.0% (median 6.2%) of the bootstrap slopes fall
below the point estimate, so they are biased upward; the basic bootstrap intervals, which correct for that, exclude zero
in 1 of 8.

### alt-transient-parallel

In [ ]:
quote("alt-transient-parallel")
rows = []
for ds in sorted(good.dataset.unique()):
    d = good[good.dataset == ds]
    tr200 = {p: rsg.production_rates(spikes, d, p, stop_ms=200) for p in ["Short", "Long"]}
    X = np.vstack([r for p in tr200 for r in tr200[p].values()])
    mu, comps, k, ev = rsg.pca_fit(X)
    for prior in ["Short", "Long"]:
        tsv = rsg.PRIOR_TS[prior]
        mid = rsg.project(tr200[prior][tsv[2]], mu, comps)
        for which, ts in [("shortest", tsv[0]), ("longest", tsv[-1])]:
            dist = np.linalg.norm(rsg.project(tr200[prior][ts], mu, comps) - mid, axis=1)
            t = 20 * np.arange(1, len(dist) + 1)
            rows.append(dict(dataset=ds, k=k, prior=prior, ts=which, slope=stats.linregress(t, dist).slope,
                             dist_first=dist[0], dist_last=dist[-1]))
ALT2 = pd.DataFrame(rows)
print(ALT2.round(4).to_string(index=False))
for which in ["shortest", "longest"]:
    s = ALT2[ALT2.ts == which].slope
    print(f"{which} ts: two-sided signed-rank of 8 slopes against 0, p = {stats.wilcoxon(s).pvalue:.3f}; "
          f"median slope {s.median():.4f} per ms; median change over 200 ms {200 * s.median():.2f} against median distance "
          f"{ALT2[ALT2.ts == which].dist_first.median():.2f}")

**Result: reproduced.** The signed-rank tests of the 8 slopes against zero give p = 0.195 for the
shortest ts and 1.0 for the longest, so the paper's test finds no change. For the longest ts the
median change over 200 ms is 0.07 spikes/s against a median distance of 22.3. For the shortest ts
the distance grows in 6 of 8 dataset-priors (median change 14.5 against a median starting distance of
18.2): a divergence, not the convergence the transient hypothesis predicts.

### alt-threshold-monotonic

Trajectories aligned to Go, from Go - 800 ms to Go, PCA per dataset at 60% of the variance (the legend's
value); the same at 75% is shown as a variant (the legend: "results remained unchanged when more PCs
were included"). Times are the ends of 20 ms bins relative to Go, so the last bin ends at 0.

In [ ]:
quote("alt-threshold-monotonic")
rows = []
curves = {}
for thr, min_k in [(0.60, 1), (0.60, 2), (0.75, 1)]:
    for ds in sorted(good.dataset.unique()):
        d = good[good.dataset == ds]
        trg = {(p, ts): rsg.aligned_rates(spikes, d[(d.prior == p) & (d.ts == ts)].go_time.values, -800, 0)
               for p in ["Short", "Long"] for ts in rsg.PRIOR_TS[p]}
        mu, comps, k, ev = rsg.pca_fit(np.vstack(list(trg.values())), thr, min_k)
        for prior in ["Short", "Long"]:
            tsv = rsg.PRIOR_TS[prior]
            mid = rsg.project(trg[(prior, tsv[2])], mu, comps)
            for ts in tsv[[0, 1, 3, 4]]:
                diff = rsg.project(trg[(prior, ts)], mu, comps) - mid
                dist = np.linalg.norm(diff, axis=1)
                j = int(np.argmin(dist))
                # with one component, a minimum where the difference changes sign is a crossing, not a convergence
                crossing = k == 1 and np.any(np.sign(diff[max(j - 1, 0):j + 2, 0]) != np.sign(diff[j, 0]))
                rows.append(dict(threshold=f"{thr:.0%}" + (", at least 2 PCs" if min_k == 2 else ""), dataset=ds, k=k, prior=prior, ts=ts,
                                 t_min=-800 + 20 * (j + 1), dist_at_go=dist[-1], dist_min=dist.min(), crossing=crossing))
                if thr == 0.60 and min_k == 1:
                    curves[(ds, prior, ts)] = dist
ALT3 = pd.DataFrame(rows)
for thr, r in ALT3.groupby("threshold", sort=False):
    print(f"variance {thr}: components per dataset {r.groupby('dataset').k.first().to_dict()}; minima at a sign crossing of a 1-component difference: {int(r.crossing.sum())}")
    print(f"  time of minimum (ms before Go): median {r.t_min.median():.0f}; in last 40 ms {int((r.t_min >= -20).sum())} of 32; "
          f"in last 200 ms {int((r.t_min >= -180).sum())} of 32; at the last bin {int((r.t_min == 0).sum())}")
    print("  ", sorted(r.t_min.tolist()))
    print("  by prior:", r.groupby("prior").t_min.median().to_dict())
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5), sharey=True)
t = np.arange(-780, 1, 20)
for (ds, prior, ts), dist in curves.items():
    axes[0 if prior == "Short" else 1].plot(t, dist, lw=0.8, alpha=0.7)
for ax, p in zip(axes, ["Short", "Long"]):
    ax.set_title(f"{p}: distance to the middle ts trajectory (60% PCs)")
    ax.set_xlabel("time to Go (ms)")
allc = np.array(list(curves.values()))
print(f"plotted distances range {allc.min():.2f} to {allc.max():.2f} (all shown)")
plt.tight_layout()
plt.show()

**Result: not reproduced.** At 60% of the variance (1 to 3 components) the median time of the minimum
distance is 230 ms before Go; the minimum falls in the last 40 ms in 4 of 32 curves and in the last
200 ms in 13. With one component (the two Eye datasets) the distance is the absolute difference of two
projections, and 12 of their 16 minima fall where that difference changes sign, a crossing, not a
convergence. Keeping at least 2 components at 60%, the median is 260 ms before Go (5 of 32 in the last
40 ms). At 75% (2 to 5 components) the median is 160 ms before Go, which would be partially reproduced.
In all three the minima spread from 740 or 760 ms before Go to Go.

## Summary

| claim | outcome |
|---|---|
| beh-slopes | not reproduced: through the origin 2 of 8 slopes below 1; with an intercept 6 of 8 |
| beh-overlap-prior | not reproduced: Short median bias at 800 ms is +28.3 ms; p = 0.073 |
| beh-sigmoid-increments | reproduced: 7 of 8, p = 0.0156 |
| beh-long-more-bias | partially reproduced: 3 of 4 through the origin (4 of 4 with an intercept) |
| beh-bayes-model | partially reproduced: SD correlation positive in 4 of 4, bias R2 >= 0.8 in 2 of 4 |
| glm-prior-ts | not reproduced: 79.2% prior-dependent, 45.8% ts-dependent |
| dfr-overlap-support | reproduced: largest difference at 480 to 640 ms, above the shuffles |
| pca-dimensionality | not reproduced: 1 of 8 counts in range (2 of 8 and 3 of 8 with narrower kernels) |
| encoding-r2 | reproduced: Hand Left 0.996 and 0.977 |
| encoding-bayes-beats-linear | partially reproduced: Short 4 of 4, Long 3 of 4 |
| short-in-long-control | reproduced (paper's test, p = 0.25), with Bayesian better in 3 of 4; ANOVA interaction p = 0.284 |
| neural-long-more-bias | partially reproduced: 3 of 4 |
| speed-vs-projection | partially reproduced: r = -0.93 and -0.87, stronger than the paper's |
| initial-speed-biased | not reproduced: 7 of 8 shallower, p = 0.055 |
| speed-predicts-tp | partially reproduced: r = -0.72 and -0.74, stronger than the paper's |
| alt-speed-unmodulated | reproduced (paper's test, p = 0.25 and 0.125); slopes positive in 7 of 8 with attrition, negative in 7 of 8 without |
| alt-transient-parallel | reproduced: p = 0.195 and 1.0 |
| alt-threshold-monotonic | not reproduced: median minimum 230 ms before Go |
| gpfa-variance-inverted-u | not reproduced: 6 of 8 negative, p = 0.055 (`gpfa_sohn_2019.ipynb`) |
| gpfa-mean-sigmoid | reproduced (`gpfa_sohn_2019.ipynb`) |
| rnn-curved-manifold, rnn-perturbation | blocked: the networks are not in the dataset nor public |
| monkey-g | blocked: monkey G is not in the dataset |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning and
the differences from the paper's method.

How the analysis differs from the paper's: it uses one session of monkey H, where the paper pools 17
sessions of H and 12 of G, so the behavioural tests have one session's power and the population tests
have 4 datasets in place of 8, which no signed-rank test over 4 values can bring below p = 0.125.
Within the session, the 283 test-file trials are unusable (no task variables), so about 80% of the
session's trials are analysed. Method details the claims file fixed are followed; the behavioural regression has no
intercept, as the Methods and Table S1 write it. Three places where the claims file may misread the paper
are asked of a reader (questions/sohn-2019.yaml): the smoothing kernel's width read as an SD (narrower
kernels would make encoding-bayes-beats-linear not reproduced), the PCA population (the paper pools 536
units of monkey H over sessions), and the z-scoring of the GPFA projections. A fourth asks whether
beh-bayes-model should compare the fit with Table S3, which the claims file took to be unavailable.
GPFA is elephant's port, not the paper's MATLAB code.

What was found about the dataset: it holds one of the paper's 29 sessions, and its test file withholds
the task variables of 283 trials and the spikes of 14 units on them. The `iti` description reverses
the two priors. `tp` exceeds `go_time - set_time` by a median of 17.9 ms, without a description of how
it is measured, so production windows ending at the shortest `tp` run past Go in 36 of 40 conditions
(ending them at Go changes no outcome). Units have no electrode assignment and no single or multi-unit label. The spikes share
the clock of the trial table.

In [ ]:
print(f"runtime {time.time() - T0:.0f} s; peak memory {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.2f} GB")